In [ ]:
# ── 0. 라이브러리 ────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path
 
import matplotlib.font_manager as fm
# 한글 폰트 자동 탐지
_korean_fonts = [f.name for f in fm.fontManager.ttflist
                 if any(k in f.name for k in ["Gothic","Nanum","Malgun","AppleGothic","NanumGothic"])]
if _korean_fonts:
    matplotlib.rcParams["font.family"] = _korean_fonts[0]
else:
    # fallback: 한글 문자를 영문으로 대체하는 대신 경고만 억제
    matplotlib.rcParams["axes.unicode_minus"] = False
matplotlib.rcParams["axes.unicode_minus"] = False
 
DATA_DIR = Path(".")   # csv 파일 위치에 맞게 수정
OUT_DIR  = Path("output")
OUT_DIR.mkdir(exist_ok=True)
 
# ── 1. 데이터 로드 ───────────────────────────────────────────────────────────
train = pd.read_csv(DATA_DIR / "/Users/Jiyeon/Desktop/ftp/data/raw/train.csv")
test  = pd.read_csv(DATA_DIR / "/Users/Jiyeon/Desktop/ftp/data/raw/test.csv")
 
print(f"Train : {train.shape}  |  Test : {test.shape}")
print(f"타겟 분포\n{train['임신 성공 여부'].value_counts()}")
print(f"임신 성공률 : {train['임신 성공 여부'].mean()*100:.1f}%")

FileNotFoundError: [Errno 2] No such file or directory: 'test.csv'

In [ ]:
# ── 2. 변수 그룹 정의 ────────────────────────────────────────────────────────
# 2-1. 순서형 횟수 컬럼 (0회 ~ 6회 이상)
COUNT_COLS = [
    "총 시술 횟수", "클리닉 내 총 시술 횟수",
    "IVF 시술 횟수", "DI 시술 횟수",
    "총 임신 횟수", "IVF 임신 횟수", "DI 임신 횟수",
    "총 출산 횟수", "IVF 출산 횟수", "DI 출산 횟수",
]
COUNT_MAP = {"0회": 0, "1회": 1, "2회": 2, "3회": 3,
             "4회": 4, "5회": 5, "6회 이상": 6}
 
# 2-2. 나이대 순서형
AGE_MAP = {
    "만18-34세": 0, "만35-37세": 1, "만38-39세": 2,
    "만40-42세": 3, "만43-44세": 4, "만45-50세": 5, "알 수 없음": -1,
}
 
# 2-3. 이진 플래그 컬럼
BINARY_COLS = [
    "배란 자극 여부", "단일 배아 이식 여부",
    "착상 전 유전 검사 사용 여부", "착상 전 유전 진단 사용 여부",
    "남성 주 불임 원인", "남성 부 불임 원인",
    "여성 주 불임 원인", "여성 부 불임 원인",
    "부부 주 불임 원인", "부부 부 불임 원인", "불명확 불임 원인",
    "불임 원인 - 난관 질환",  "불임 원인 - 남성 요인",
    "불임 원인 - 배란 장애",  "불임 원인 - 여성 요인",
    "불임 원인 - 자궁경부 문제", "불임 원인 - 자궁내막증",
    "불임 원인 - 정자 농도",  "불임 원인 - 정자 면역학적 요인",
    "불임 원인 - 정자 운동성", "불임 원인 - 정자 형태",
    "동결 배아 사용 여부", "신선 배아 사용 여부",
    "기증 배아 사용 여부",  "대리모 여부",
    "PGD 시술 여부", "PGS 시술 여부",
]
 
# 2-4. 연속형 수치 컬럼
NUMERIC_COLS = [
    "총 생성 배아 수", "미세주입된 난자 수", "미세주입에서 생성된 배아 수",
    "이식된 배아 수", "미세주입 배아 이식 수",
    "저장된 배아 수", "미세주입 후 저장된 배아 수",
    "해동된 배아 수", "해동 난자 수",
    "수집된 신선 난자 수", "저장된 신선 난자 수",
    "혼합된 난자 수", "파트너 정자와 혼합된 난자 수", "기증자 정자와 혼합된 난자 수",
    "난자 채취 경과일", "난자 혼합 경과일", "배아 이식 경과일",
    # 아래 두 컬럼은 결측 99% → 결측 여부 플래그로만 사용
    # "임신 시도 또는 마지막 임신 경과 연수",  # 결측 96%
    # "난자 해동 경과일",   # 결측 99%
    # "배아 해동 경과일",   # 결측 84%
]
 
# 2-5. 고결측(≥80%) 컬럼 → 값 제거, 결측 여부 플래그만 생성
HIGH_MISSING_COLS = [
    "임신 시도 또는 마지막 임신 경과 연수",   # 96%
    "난자 해동 경과일",                        # 99%
    "배아 해동 경과일",                        # 84%
    "착상 전 유전 검사 사용 여부",             # 99% (착상 전 유전 진단과 겹침)
    "PGD 시술 여부",                           # 99%
    "PGS 시술 여부",                           # 99%
]

In [ ]:
# ── 3. EDA ──────────────────────────────────────────────────────────────────
 
# 3-1. 결측치 히트맵
fig, ax = plt.subplots(figsize=(18, 5))
missing_pct = (train.isnull().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
ax.bar(range(len(missing_pct)), missing_pct.values, color="#3266ad")
ax.set_xticks(range(len(missing_pct)))
ax.set_xticklabels(missing_pct.index, rotation=75, ha="right", fontsize=9)
ax.set_ylabel("결측률 (%)")
ax.set_title("변수별 결측률")
ax.axhline(80, color="red", linestyle="--", linewidth=1, label="80% 기준선")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "01_missing_rate.png", dpi=120)
plt.close()
 
# 3-2. 타겟 분포
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
 
# 전체
axes[0].pie([190123, 66228], labels=["실패(0)", "성공(1)"],
            colors=["#3266ad","#1D9E75"], autopct="%1.1f%%", startangle=90)
axes[0].set_title("임신 성공 여부 분포")
 
# 나이대별 성공률
age_rate = train.groupby("시술 당시 나이")["임신 성공 여부"].mean().reindex(
    ["만18-34세","만35-37세","만38-39세","만40-42세","만43-44세","만45-50세"])
age_rate.plot(kind="bar", ax=axes[1], color="#7F77DD", edgecolor="none")
axes[1].set_title("나이대별 임신 성공률")
axes[1].set_ylabel("성공률")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(OUT_DIR / "02_target_age.png", dpi=120)
plt.close()
 
# 3-3. 시술 유형별 성공률
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
# 시술 유형 (IVF/DI)
type_rate = train.groupby("시술 유형")["임신 성공 여부"].mean()
type_rate.plot(kind="bar", ax=axes[0], color=["#E24B4A","#3266ad"], edgecolor="none")
axes[0].set_title("시술 유형별 성공률")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)
 
# 특정 시술 유형 top10
top10_types = train["특정 시술 유형"].value_counts().head(10).index
rate_by_type = train[train["특정 시술 유형"].isin(top10_types)] \
    .groupby("특정 시술 유형")["임신 성공 여부"].mean().sort_values()
rate_by_type.plot(kind="barh", ax=axes[1], color="#378ADD", edgecolor="none")
axes[1].set_title("특정 시술 유형별 성공률 (Top 10)")
plt.tight_layout()
plt.savefig(OUT_DIR / "03_treatment_type.png", dpi=120)
plt.close()
 
# 3-4. 수치형 변수 분포 (성공/실패 비교)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
plot_num_cols = [
    "총 생성 배아 수", "이식된 배아 수", "수집된 신선 난자 수",
    "혼합된 난자 수", "저장된 배아 수", "미세주입된 난자 수",
    "난자 채취 경과일", "배아 이식 경과일",
]
for ax, col in zip(axes.flat, plot_num_cols):
    for label, color in [(0, "#3266ad"), (1, "#1D9E75")]:
        grp = train[train["임신 성공 여부"] == label][col].dropna()
        clip_val = grp.quantile(0.99)
        grp = grp.clip(upper=clip_val)
        if grp.nunique() > 1:
            grp.plot.kde(ax=ax, label=f"{'성공' if label else '실패'}",
                         linewidth=1.5, color=color)
        else:
            ax.axvline(grp.iloc[0] if len(grp) > 0 else 0, color=color,
                       label=f"{'성공' if label else '실패'}", linewidth=1.5)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel("")
plt.suptitle("수치형 변수 KDE (성공 vs 실패)", y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "04_numeric_kde.png", dpi=120)
plt.close()
 
# 3-5. 이진 변수별 성공률
flag_rates = {col: train.groupby(col)["임신 성공 여부"].mean() for col in BINARY_COLS}
fig, ax = plt.subplots(figsize=(14, 6))
rate_1 = pd.Series({col: train[train[col] == 1]["임신 성공 여부"].mean()
                    for col in BINARY_COLS})
rate_0 = pd.Series({col: train[train[col] == 0]["임신 성공 여부"].mean()
                    for col in BINARY_COLS})
diff = (rate_1 - rate_0).sort_values()
diff.plot(kind="barh", ax=ax, color=diff.map(lambda x: "#1D9E75" if x > 0 else "#E24B4A"))
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("이진 변수 1일 때 vs 0일 때 성공률 차이")
ax.set_xlabel("성공률 차이 (1그룹 - 0그룹)")
plt.tight_layout()
plt.savefig(OUT_DIR / "05_binary_effect.png", dpi=120)
plt.close()
 
# 3-6. 시술 횟수별 성공률
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, col in zip(axes.flat, COUNT_COLS):
    rate = train.groupby(col)["임신 성공 여부"].mean().reindex(COUNT_MAP.keys())
    rate.plot(kind="bar", ax=ax, color="#7F77DD", edgecolor="none")
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
plt.suptitle("시술/임신/출산 횟수별 성공률", y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "06_count_cols_rate.png", dpi=120)
plt.close()
 
print("✅ EDA 차트 저장 완료 →", OUT_DIR)
 

In [ ]:
# ── 4. 전처리 ────────────────────────────────────────────────────────────────
 
def preprocess(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    df = df.copy()
 
    # ── 4-1. ID 제거
    df = df.drop(columns=["ID"])
 
    # ── 4-2. 고결측 컬럼: 값 → 결측 여부 플래그 (is_missing_*)
    for col in HIGH_MISSING_COLS:
        if col in df.columns:
            df[f"결측_{col}"] = df[col].isna().astype(np.int8)
            df = df.drop(columns=[col])
 
    # ── 4-3. 순서형 횟수 컬럼 → 정수
    for col in COUNT_COLS:
        df[col] = df[col].map(COUNT_MAP).astype(float)
 
    # ── 4-4. 나이대 → 순서형 정수 (-1 = 알 수 없음)
    df["시술 당시 나이"] = df["시술 당시 나이"].map(AGE_MAP).astype(float)
 
    # ── 4-5. 시술 시기 코드 → 레이블 인코딩 (빈도 기반 순서)
    period_order = ["TRDQAZ","TRCMWS","TRYBLT","TRVNRY","TRJXFG","TRZKPL","TRXQMD"]
    period_map = {v: i for i, v in enumerate(period_order)}
    df["시술 시기 코드"] = df["시술 시기 코드"].map(period_map).astype(float)
 
    # ── 4-6. 시술 유형 → 이진 (IVF=1, DI=0)
    df["시술 유형_IVF"] = (df["시술 유형"] == "IVF").astype(np.int8)
    df = df.drop(columns=["시술 유형"])
 
    # ── 4-7. 특정 시술 유형 → 핵심 시술명 추출 (멀티-레이블 → 이진 플래그)
    # "/" 또는 ":" 구분자로 복합 시술을 분해
    procedure_types = ["IVF","ICSI","IUI","ICI","GIFT","FER","DI","BLASTOCYST","AH","Unknown"]
    for pt in procedure_types:
        df[f"시술_{pt}"] = df["특정 시술 유형"].fillna("").str.contains(pt, regex=False).astype(np.int8)
    df = df.drop(columns=["특정 시술 유형"])
 
    # ── 4-8. 배아 생성 주요 이유 → 멀티-레이블 이진화
    reason_types = ["현재 시술용", "배아 저장용", "난자 저장용", "기증용", "연구용"]
    for rt in reason_types:
        df[f"이유_{rt}"] = df["배아 생성 주요 이유"].fillna("").str.contains(rt, regex=False).astype(np.int8)
    df = df.drop(columns=["배아 생성 주요 이유"])
 
    # ── 4-9. 난자/정자 출처 → 원-핫 인코딩
    egg_dummies  = pd.get_dummies(df["난자 출처"],  prefix="난자출처",  dtype=np.int8)
    sperm_dummies = pd.get_dummies(df["정자 출처"], prefix="정자출처",  dtype=np.int8)
    df = pd.concat([df.drop(columns=["난자 출처","정자 출처"]), egg_dummies, sperm_dummies], axis=1)
 
    # ── 4-10. 기증자 나이 → 순서형 정수
    donor_age_map = {
        "만20세 이하": 0, "만21-25세": 1, "만26-30세": 2,
        "만31-35세": 3, "만36-40세": 4, "만41-45세": 5, "알 수 없음": -1,
    }
    for col in ["난자 기증자 나이", "정자 기증자 나이"]:
        df[col] = df[col].map(donor_age_map).astype(float)
 
    # ── 4-11. 배란 유도 유형 → 원-핫 인코딩
    induction_dummies = pd.get_dummies(df["배란 유도 유형"], prefix="배란유도", dtype=np.int8)
    df = pd.concat([df.drop(columns=["배란 유도 유형"]), induction_dummies], axis=1)
 
    # ── 4-12. 이진 플래그 컬럼 → int8 (결측은 0으로 대체)
    binary_in_df = [c for c in BINARY_COLS if c in df.columns]
    df[binary_in_df] = df[binary_in_df].fillna(0).astype(np.int8)
 
    # ── 4-13. 연속형 수치 컬럼 → 중앙값 대체
    num_in_df = [c for c in NUMERIC_COLS if c in df.columns]
    for col in num_in_df:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
 
    # ── 4-14. 파생 피처 생성
    # 배아 활용률: 이식 배아 수 / 총 생성 배아 수
    df["배아_활용률"] = np.where(
        df["총 생성 배아 수"] > 0,
        df["이식된 배아 수"] / df["총 생성 배아 수"], 0
    )
    # 수정률: 총 생성 배아 수 / 혼합된 난자 수
    df["배아_수정률"] = np.where(
        df["혼합된 난자 수"] > 0,
        df["총 생성 배아 수"] / df["혼합된 난자 수"], 0
    )
    # 총 불임 원인 수
    infertility_cols = [c for c in df.columns if "불임 원인" in c]
    df["불임원인_총수"] = df[infertility_cols].sum(axis=1)
 
    # ── 4-15. 타겟 분리 (train만)
    if is_train:
        y = df["임신 성공 여부"].astype(np.int8)
        df = df.drop(columns=["임신 성공 여부"])
        return df, y
 
    return df
 
 


In [ ]:
# ── 5. 전처리 실행 ────────────────────────────────────────────────────────────
X_train, y_train = preprocess(train, is_train=True)
X_test  = preprocess(test,  is_train=False)
 
# 컬럼 정합성 맞추기 (one-hot 후 train/test 컬럼 차이 보정)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)
 
print(f"\n전처리 후 Train shape : {X_train.shape}")
print(f"전처리 후 Test  shape : {X_test.shape}")
print(f"\n최종 피처 수 : {X_train.shape[1]}")
print(f"\n샘플 피처 목록:\n{list(X_train.columns[:20])}")

In [ ]:
# ── 6. 저장 ─────────────────────────────────────────────────────────────────
X_train.to_csv(OUT_DIR / "X_train.csv", index=False)
y_train.to_csv(OUT_DIR / "y_train.csv", index=False, header=True)
X_test.to_csv(OUT_DIR  / "X_test.csv",  index=False)
 
print(f"\n✅ 전처리된 CSV 저장 완료 → {OUT_DIR}/")

In [ ]:
# ── 7. 상관관계 히트맵 (파생 피처 포함 수치형) ──────────────────────────────
numeric_final = X_train.select_dtypes(include=[np.number]).columns.tolist()
# 상위 30개만 (상관관계 절댓값 기준)
corr_with_target = pd.concat([X_train[numeric_final], y_train.rename("임신 성공 여부")], axis=1) \
    .corr()["임신 성공 여부"].drop("임신 성공 여부").abs().sort_values(ascending=False)
top30 = corr_with_target.head(30).index.tolist()
 
fig, ax = plt.subplots(figsize=(14, 12))
corr_matrix = pd.concat([X_train[top30], y_train.rename("임신 성공 여부")], axis=1).corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap="RdBu_r", center=0,
            annot=False, linewidths=0.5, ax=ax, vmin=-0.5, vmax=0.5)
ax.set_title("피처 상관관계 히트맵 (타겟 상관 상위 30)")
plt.tight_layout()
plt.savefig(OUT_DIR / "07_correlation_heatmap.png", dpi=120)
plt.close()
print("✅ 상관관계 히트맵 저장 완료")
 
print("\n" + "="*60)
print("전처리 요약")
print("="*60)
print(f"  원본 피처 수   : {train.shape[1] - 2}  (ID, 타겟 제외)")
print(f"  최종 피처 수   : {X_train.shape[1]}")
print(f"  타겟 성공률    : {y_train.mean()*100:.1f}%  (불균형 주의)")
print(f"  결측 잔여 여부 : {X_train.isnull().sum().sum()} 개")
print("="*60)
 